# Runbook: bet sizing from predicted probabilities (SYNTHETIC data)

> **Every number in this notebook is computed on SYNTHETIC data.** The main study uses price paths
> simulated with a **planted, documented signal** (the generator of runbook 11), so the right answer
> is known in advance. The control study uses the committed sample read by `openquant.data.fetch`
> (`SYN_A` to `SYN_E`, seeded log-normal random walks, see `DATA_SOURCES.md`), on which no rule can
> have an edge. Nothing here describes a real market.

AFML Chapter 10 turns a classifier's probability into a position: Snippet 10.1 maps the probability
of the predicted class to a size through a $z$-statistic and the normal CDF, Snippet 10.2 averages
the sizes of the bets that are live at the same time, and Snippet 10.3 rounds the result to a step
so that small changes do not trade. Runbook 11 (`11_meta_labeling_triple_barrier.ipynb`, #47)
found, **after looking at its grid**, that sizing meta-labeled bets this way had the best net
Sharpe ratio on its headline path (1.28, DSR 1.000) and gained 0.36 of annualised Sharpe ratio over
the unfiltered primary model in its Monte Carlo, more than twice what a flat
$p \ge 0.5$ filter gained. It flagged that as a candidate, not a result.

This runbook is the **pre-registered test of that candidate**. It reuses runbook 11's generator,
primary model, meta-model and purged-CV pipeline unchanged, on **fresh seeds**, and asks whether
probability sizing beats the flat filter it was compared with, whether the gain survives deflation
by every sizing choice tried (number of classes, step size, averaging), and whether it invents an
edge where there is none. It reports turnover, drawdown and the cost at which each rule stops
paying, so the answer is not only a Sharpe ratio at one cost.

**Run it on your own data.** The control study reads three environment variables, so it runs
unchanged on any daily OHLCV file `openquant.data` can load. Write the executed copy and figures
outside the repository:

```bash
OPENQUANT_RUNBOOK_SOURCE=/path/to/daily_ohlcv.parquet \
OPENQUANT_RUNBOOK_SYMBOLS=ES,NQ,CL \
OPENQUANT_RUNBOOK_RANGE=2010-01-01:2024-12-31 \
OPENQUANT_RUNBOOK_REGISTRY=$HOME/.openquant/trials-bet-sizing.json \
OPENQUANT_FIGURE_DIR=/tmp/bet-sizing-figures \
  uv run --python .venv/bin/python python notebooks/python/scripts/execute_notebook_cells.py \
    notebooks/python/13_bet_sizing_from_probabilities.ipynb --out /tmp/13_on_my_data.ipynb
```

`OPENQUANT_RUNBOOK_REGISTRY` keeps the control's trial registry between runs, so every
configuration you try on your data counts toward its deflated Sharpe ratio. For a vendor API, set
`SOURCE` in the parameters cell to a `CallableSource`. The planted-signal study does not depend on
the data source.

Sections follow the research-notebook contract (#46): Setup, Hypothesis, Data, Method, Results,
Analysis, Promotion decision, Self-review, Reproducibility.

## Setup

In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import sys
import tempfile
import time
from datetime import date, timedelta
from importlib.metadata import version
from pathlib import Path

import nbfigures
import numpy as np
import openquant
import polars as pl
from openquant import backtest_stats, bet_sizing, evaluation, labeling, sampling, volatility
from openquant.cross_validation import (
    count_train_test_overlaps,
    naive_kfold_splits,
    purged_kfold_splits,
    split_with_diagnostics,
)

# ---- planted-signal generator: runbook 11's, unchanged; only the seed is new ---------------------
SEED = 50  # runbook 11 used 47; every path here is new
N_BARS = 5000  # business days per simulated path, about 20 years
P_STAY = 0.99  # regime persistence: spells last 100 bars on average
VOL_TREND, VOL_CHOP = 0.007, 0.014  # daily volatility in each regime
LOGVOL_AR, LOGVOL_SD = 0.97, 0.05  # AR(1) noise on log volatility: vol is an imperfect regime proxy
MOM_LEN = 20  # the move the drift follows (trend) or fades (chop)
KAPPA_TREND, KAPPA_CHOP = 0.25, -0.10  # headline signal strength (runbook 11's calibration)
STRENGTHS = (0.0, 0.12, 0.25)  # Monte Carlo sweep of KAPPA_TREND, with KAPPA_CHOP = -0.4 x it
N_REPS = int(os.environ.get("OPENQUANT_RUNBOOK_REPS", "30"))  # paths per strength

# ---- meta-labeling pipeline: runbook 11's headline configuration, unchanged ----------------------
VOL_SPAN = 50  # volatility.get_daily_vol EWM span (Snippet 3.1)
CUSUM_MULT = 2.0  # CUSUM threshold: 2 x the daily vol known at the previous bar (Snippet 2.4)
PT_SL = 1.5  # horizontal barriers at +-1.5 x daily vol (Snippet 3.2)
HORIZON_DAYS = 14  # vertical barrier, calendar days (Snippet 3.4)
FAST, SLOW = 10, 40  # primary model: moving-average crossover on the log close
N_SPLITS, PCT_EMBARGO = 5, 0.01  # purged k-fold (Snippet 7.3)
L2 = 1.0  # ridge penalty of the logistic meta-model, on standardised features
FEATURES = ("vol_ratio", "efficiency", "ma_gap")

# ---- bet sizing (this runbook), fixed before the first run ----------------------------------------
COST_BPS = 5.0  # cost per unit of one-way turnover, in basis points (as runbook 11)
COST_GRID_BPS = (0.0, 2.0, 5.0, 10.0, 20.0, 40.0)  # cost sensitivity, reported, not selected on
NUM_CLASSES = (2, 3)  # K in Snippet 10.1; 2 is right for a binary meta-model
STEP_SIZES = (0.0, 0.05, 0.1, 0.2)  # Snippet 10.3 discretisation; 0 = none
HEADLINE = {"sizing": "prob", "k": 2, "step": 0.1, "avg": True, "cv": "purged-kfold"}
FILTER = {"sizing": "binary", "k": None, "step": None, "avg": True, "cv": "purged-kfold"}
PRIMARY = {"sizing": "fixed", "k": None, "step": None, "avg": True, "cv": None}
N_SHUFFLES = 200  # permutations of the probabilities in the shuffle test

# ---- control data source ------------------------------------------------------------------------
# Default: the committed SYNTHETIC sample (no network). Override with the env vars below, or set
# SOURCE to any openquant.data DataSource (e.g. a CallableSource around your vendor's API).
_src_env = os.environ.get("OPENQUANT_RUNBOOK_SOURCE", "").strip()
SOURCE = openquant.data.LocalFileSource(_src_env, name="user-file") if _src_env else None
SYMBOLS = os.environ.get("OPENQUANT_RUNBOOK_SYMBOLS", "SYN_A,SYN_B,SYN_C,SYN_D,SYN_E").split(",")
START, END = os.environ.get("OPENQUANT_RUNBOOK_RANGE", "2022-01-03:2023-12-29").split(":")
SYNTHETIC = SOURCE is None
LABEL = "SYNTHETIC" if SYNTHETIC else "user data"
_reg_env = os.environ.get("OPENQUANT_RUNBOOK_REGISTRY", "").strip()
REGISTRY_DIR = Path(tempfile.mkdtemp(prefix="oq-nb13-"))

print(
    f"planted signal: kappa_trend={KAPPA_TREND}, kappa_chop={KAPPA_CHOP}, {N_BARS} bars, seed {SEED}"
)
print(f"Monte Carlo: strengths {STRENGTHS} x {N_REPS} paths")
print(f"headline sizing rule: {HEADLINE}")
print(
    f"control source: {'committed SYNTHETIC sample' if SYNTHETIC else 'user file'}; symbols {SYMBOLS}"
)
# Versions change with dependency bumps, not with the study, so they go to stderr (see #148/#149).
print(f"numpy {np.__version__} | polars {pl.__version__}", file=sys.stderr)

## Hypothesis

Written down before the first full run of this notebook; the parameters cell above was fixed at the
same time. **Filter** is runbook 11's pre-registered rule: take the primary model's bet at full
size when the meta-model's out-of-fold $P(\text{profitable}) \ge 0.5$, else stand aside.
**Sized** is the headline sizing rule: Snippet 10.1 with $K = 2$ classes on the same probability
(size $2\Phi(z) - 1$ in the primary model's direction when $p \ge 0.5$, zero otherwise), the sizes
of concurrent bets averaged (Snippet 10.2), and the average rounded to steps of 0.1 (Snippet 10.3),
all through `bet_sizing.bet_size_probability`. Both use the same probabilities and pay 5 bps per
unit of one-way turnover.

- **H1 (sizing beats filtering).** At the headline strength ($\kappa_{\text{trend}} = 0.25$),
  sized's annualised net Sharpe ratio is higher than filter's: paired one-sided $t > 1.645$ over 30
  fresh simulated paths.
- **H2 (evidence on one path).** On a fresh headline path, sized's deflated Sharpe ratio is at
  least 0.95, deflated by **every** configuration evaluated on that path (the trial grid below: 21
  configurations, including the number-of-classes, step-size and averaging choices, and
  configurations that take no bet).
- **H3 (no false discovery).** Where there is no signal, sizing does not manufacture one:
  - (a) on the zero-signal generator ($\kappa = 0$, 30 paths), sized does not beat filter
    (paired $t < 1.645$);
  - (b) on the same paths, the *best* configuration of the 21-trial grid, chosen after the fact
    on each path, has a DSR of at least 0.95 on no more than 5% of paths (exact one-sided binomial
    test at 5%: 5 or more of 30 fails);
  - (c) on the `SYN_*` sample, neither sized nor the best of the grid has a DSR of at least 0.95.

**Secondary, reported but not tested:** the same comparison at $\kappa_{\text{trend}} = 0.12$;
sized against the unfiltered primary model; walk-forward versions; turnover, maximum drawdown and
the break-even cost of each rule; and every grid configuration's Monte Carlo mean.

**What would count against the candidate.** Runbook 11's gain was measured against the primary
model. Part of it is filtering (the sized rule also stands aside below $p = 0.5$), so the question
here is the *extra* gain from sizing over filtering. If H1 fails, the post hoc result was mostly the
filter; if H2 fails, one 20-year path cannot tell sizing apart from luck among 21 tries; if H3 fails,
sizing amplifies noise.

## Data

### Planted-signal generator (runbook 11's, fresh seeds)

A two-state Markov regime $s_t \in \{\text{trend}, \text{chop}\}$ stays put with probability
0.99. Daily volatility is $\sigma_t = \bar\sigma_{s_t} e^{v_t}$ with $\bar\sigma = 0.7\%$ (trend),
$1.4\%$ (chop) and $v_t$ an AR(1) with coefficient 0.97 and shock s.d. 0.05. Log returns are

$$r_t = \kappa_{s_t}\,\sigma_t\,\tanh\!\Big(\frac{\sum_{k=1}^{20} r_{t-k}}{\sigma_t\sqrt{20}}\Big) + \sigma_t\,\varepsilon_t,
\qquad \varepsilon_t \sim N(0,1),$$

with $\kappa_{\text{trend}} = 0.25$ and $\kappa_{\text{chop}} = -0.10$ at the headline strength. The
generator, its calibration and its disclosure are runbook 11's; nothing about it was changed here.
Runbook 11 raised the strength during its own prototypes until a meta-labeling effect was
detectable, which favours finding *some* effect of the meta-model; it says nothing about whether
sizing beats filtering, the question here.

**Fresh seeds.** Runbook 11 found the candidate on its headline path (seed `[47, 0]`) and its Monte
Carlo paths (`[47, k + 1, path]`). Testing on those paths would re-use the data the candidate was
selected on, so this notebook uses seed 50 throughout: headline path `[50, 0]`, Monte Carlo paths
`[50, k + 1, path]`.

The code below is copied from runbook 11 without changes, apart from the seed.

In [ ]:
# ---- copied from 11_meta_labeling_triple_barrier.ipynb (#47 / PR #181), unchanged -----------------
def business_days(n: int, start: date = date(2000, 1, 3)) -> list[str]:
    out, d = [], start
    while len(out) < n:
        if d.weekday() < 5:
            out.append(f"{d.isoformat()} 00:00:00")
        d += timedelta(days=1)
    return out


def simulate(seed, kappa_trend: float, kappa_chop: float, n: int = N_BARS):
    """Close prices and the hidden regime (True = trend) of one planted-signal path."""
    rng = np.random.default_rng(seed)
    trend = np.empty(n, dtype=bool)
    trend[0] = rng.random() < 0.5
    stay = rng.random(n) < P_STAY
    for t in range(1, n):
        trend[t] = trend[t - 1] if stay[t] else not trend[t - 1]
    shock = rng.standard_normal(n)
    logvol = np.zeros(n)
    for t in range(1, n):
        logvol[t] = LOGVOL_AR * logvol[t - 1] + LOGVOL_SD * shock[t]
    sigma = (np.where(trend, VOL_TREND, VOL_CHOP) * np.exp(logvol)).tolist()
    eps = rng.standard_normal(n).tolist()
    regime = trend.tolist()
    r = [0.0] * n
    move = 0.0  # sum of the last MOM_LEN returns, all strictly before t
    for t in range(1, n):
        kappa = kappa_trend if regime[t] else kappa_chop
        r[t] = (
            kappa * sigma[t] * math.tanh(move / (sigma[t] * math.sqrt(MOM_LEN))) + sigma[t] * eps[t]
        )
        move += r[t] - (r[t - MOM_LEN] if t >= MOM_LEN else 0.0)
    return 100.0 * np.exp(np.cumsum(r)), trend


# ---- end of copy -----------------------------------------------------------------------------------

TS = business_days(N_BARS)
close_h, regime_h = simulate([SEED, 0], KAPPA_TREND, KAPPA_CHOP)
planted = {"PLANTED": (TS, close_h)}
planted_hash = openquant.data.dataset_hash(pl.DataFrame({"close": close_h, "trend": regime_h}))
print(f"headline path: {N_BARS} bars, {TS[0][:10]} to {TS[-1][:10]} (SYNTHETIC calendar)")
print(
    f"share of bars in the trend regime: {regime_h.mean():.3f}; "
    f"regime spells: {1 + int(np.sum(regime_h[1:] != regime_h[:-1]))}"
)
# Float bits of the simulated path can differ in the last place between platforms (libm), so the
# hash goes to stderr, which the staleness check ignores; every number derived from it is checked.
print("planted path dataset_hash:", planted_hash, file=sys.stderr)

### Control: the SYNTHETIC `fetch` sample

In [ ]:
bars, meta = openquant.data.fetch(SYMBOLS, START, END, source=SOURCE, return_meta=True)
print("source:", meta["source"], f"| {LABEL}" + (", not market data" if SYNTHETIC else ""))
print("rows:", meta["rows"], "| dataset_hash:", meta["dataset_hash"])

control = {}
for sym in SYMBOLS:
    frame = bars.filter(pl.col("symbol") == sym).sort("ts")
    control[sym] = (
        [t.strftime("%Y-%m-%d %H:%M:%S") for t in frame["ts"].to_list()],
        frame["close"].to_numpy().astype(float),
    )
pl.DataFrame(
    {
        "symbol": SYMBOLS,
        "bars": [len(control[s][1]) for s in SYMBOLS],
        "first": [control[s][0][0][:10] for s in SYMBOLS],
        "last": [control[s][0][-1][:10] for s in SYMBOLS],
    }
)

## Method

**Probabilities (runbook 11, unchanged).** Per series: daily volatility (`volatility.get_daily_vol`,
Snippet 3.1); CUSUM events at $2\sigma_{t-1}$ (Snippet 2.4, in numpy because
`filters.cusum_filter_*` takes only a scalar threshold from Python; checked against
`cusum_filter_indices`); the side of a 10/40 moving-average crossover; triple-barrier meta-labels
at $\pm 1.5\sigma_{t_0}$ with a 14-day vertical barrier (`labeling.add_vertical_barrier`,
`triple_barrier_events`, `get_bins`); three past-only features; average-uniqueness weights
(`sampling.get_av_uniqueness_from_triple_barrier`); and an L2 logistic meta-model whose
out-of-fold probabilities $p$ come from `cross_validation.purged_kfold_splits(t0, t1, 5,
pct_embargo=0.01)`. Walk-forward keeps only training events before the test fold. Every sizing
rule below uses the **same** probabilities, so the rules differ only in how they turn $p$ into a
position.

**From probability to position (this runbook).** Each event is a bet on the primary model's side
$s$, live on $[t_0, t_1)$, where $t_1$ is the first barrier touch.

| Rule | Size of one bet | AFML |
|---|---|---|
| `fixed` (primary) | $s$ | — |
| `binary` (filter) | $s \cdot 1\{p \ge 0.5\}$ | §3.6 |
| `prob` | $s \cdot 1\{p \ge 0.5\} \cdot (2\Phi(z) - 1)$, $z = \dfrac{p - 1/K}{\sqrt{p(1-p)}}$ | 10.1 |
| `prob-signed` | $s \cdot (2\Phi(z) - 1)$ with $K = 2$: *against* the primary model when $p < 0.5$ | 10.1 as written |

Then, per series:

- **Averaging (Snippet 10.2).** With `avg`, the position at each bar is the mean size of the bets
  live at that bar, skipped bets counting as 0 (runbook 11's convention). With `latest`, it is the
  size of the most recently opened live bet: each new signal replaces the old one, which is what
  averaging is meant to prevent.
- **Discretisation (Snippet 10.3).** The position is rounded to the nearest multiple of the step
  (0 = none). Rounding is to the nearest step, so a step of 0.2 also zeroes positions below 0.1.
- `prob` rules are computed by `bet_sizing.bet_size_probability(starts, ends, p', s', K, step,
  average_active)` with $p' = \max(p, 1-p)$ and $s' = s \cdot 1\{p \ge 0.5\}$, which is
  `get_signal` → `avg_active_signals` → `discrete_signal`, the book's order. The notebook asserts
  that it equals that composition.
- $K$ sets the no-skill probability $1/K$. For a binary meta-model $K = 2$ is correct; $K = 3$
  moves the zero point to $1/3$, so every bet with $p \ge 0.5$ gets at least a 0.26 position. It is
  in the grid because it is the knob the API exposes and a researcher tuning the rule would try it.

**Returns and costs.** A position set at the close of bar $t$ earns the return of bar $t + 1$:
$w_{t-1} r_t - c\,|w_t - w_{t-1}|$, with $c$ = 5 bps per unit of one-way turnover, from the first
event bar on. Sharpe ratios are daily (as `openquant.evaluation` computes them), with
$\times\sqrt{252}$ versions for reading. **Break-even cost** is the cost per unit turnover at which
the mean net return is zero: mean gross return / mean turnover. It is the simplest capacity
measure available on a single asset: a rule with more turnover has less room for market impact.
Maximum drawdown is from `backtest_stats.drawdown_and_time_under_water` on the net equity curve.

**Deflation.** Every configuration evaluated on a path is recorded before any deflated Sharpe
ratio is read: on the headline path in an `evaluation.TrialRegistry`, whose trials the DSR deflates
by; on Monte Carlo paths by passing every trial's Sharpe ratio to
`evaluation.deflated_sharpe_ratio`. `TrialRegistry.record` rejects constant returns (#187; its fix,
PR #190, is not merged on this base), so a configuration that takes no bet is counted with a Sharpe
ratio of 0 instead, as runbook 11 did.

In [ ]:
# ---- copied from 11_meta_labeling_triple_barrier.ipynb (#47 / PR #181), unchanged -----------------
def rolling_mean(x: np.ndarray, w: int) -> np.ndarray:
    c = np.cumsum(np.r_[0.0, x])
    out = np.full(len(x), np.nan)
    out[w - 1 :] = (c[w:] - c[:-w]) / w
    return out


def ewm_mean(x: np.ndarray, span: int) -> np.ndarray:
    a, out, m = 2.0 / (span + 1.0), np.empty(len(x)), x[0]
    for i, v in enumerate(x):
        m = v if i == 0 else a * v + (1.0 - a) * m
        out[i] = m
    return out


def primary_and_features(close: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Side of the crossover and the meta features at every bar, from closes up to that bar."""
    logp = np.log(close)
    r = np.r_[0.0, np.diff(logp)]
    ma_fast, ma_slow = rolling_mean(logp, FAST), rolling_mean(logp, SLOW)
    side = np.sign(ma_fast - ma_slow)
    vol20, vol250 = np.sqrt(ewm_mean(r**2, 20)), np.sqrt(ewm_mean(r**2, 250))
    with np.errstate(divide="ignore", invalid="ignore"):
        efficiency = np.full(len(close), np.nan)
        efficiency[20:] = np.abs(logp[20:] - logp[:-20]) / (rolling_mean(np.abs(r), 20)[20:] * 20)
        feats = np.column_stack(
            [
                np.log(vol20 / vol250),
                efficiency,
                np.abs(ma_fast - ma_slow) / (vol20 * math.sqrt(SLOW)),
            ]
        )
    feats[: SLOW + 20] = np.nan  # warm-up
    return side, feats


def cusum_events(close: np.ndarray, threshold: np.ndarray) -> np.ndarray:
    """Symmetric CUSUM on log returns with one threshold per bar (Snippet 2.4)."""
    lr = np.diff(np.log(close)).tolist()
    th = threshold.tolist()
    s_pos = s_neg = 0.0
    out = []
    for i in range(1, len(close)):
        if not math.isfinite(th[i]):
            continue
        s_pos, s_neg = max(0.0, s_pos + lr[i - 1]), min(0.0, s_neg + lr[i - 1])
        if s_neg < -th[i]:
            s_neg = 0.0
            out.append(i)
        elif s_pos > th[i]:
            s_pos = 0.0
            out.append(i)
    return np.asarray(out, dtype=np.int64)


def daily_vol(ts: list[str], close: np.ndarray) -> np.ndarray:
    got = dict(volatility.get_daily_vol(ts, close.tolist(), VOL_SPAN))
    return np.array([got.get(t, np.nan) for t in ts])


def build_events(series: dict, cusum_mult: float = CUSUM_MULT) -> dict:
    """Events of every series, pooled in time order: spans, sides, meta-labels, features, weights."""
    parts = []
    for sym, (ts, close) in series.items():
        sigma = daily_vol(ts, close)
        side, feats = primary_and_features(close)
        idx = cusum_events(close, cusum_mult * np.r_[np.nan, sigma[:-1]])
        idx = idx[np.isfinite(sigma[idx]) & np.isfinite(feats[idx]).all(axis=1) & (side[idx] != 0)]
        t_events = [ts[i] for i in idx]
        vertical = labeling.add_vertical_barrier(
            t_events, ts, close.tolist(), num_days=HORIZON_DAYS
        )
        at = {t: i for i, t in enumerate(ts)}
        found = labeling.triple_barrier_events(
            ts,
            close.tolist(),
            [t for t, _ in vertical],
            ts,
            sigma.tolist(),
            pt=PT_SL,
            sl=PT_SL,
            min_ret=0.0,
            vertical_barrier_times=vertical,
            side_prediction=[(t, float(side[at[t]])) for t, _ in vertical],
        )
        bins = labeling.get_bins(found, ts, close.tolist())
        assert [e[0] for e in found] == [b[0] for b in bins]
        t0 = np.array([at[e[0]] for e in found], dtype=np.int64)
        t1 = np.array([at[e[1]] for e in found], dtype=np.int64)
        spans = list(zip(t0.tolist(), t1.tolist()))
        parts.append(
            {
                "symbol": np.full(len(t0), sym),
                "t0": t0,
                "t1": t1,
                "t0_time": np.array([ts[i] for i in t0], dtype="datetime64[ns]"),
                "t1_time": np.array([ts[i] for i in t1], dtype="datetime64[ns]"),
                "side": side[t0],
                "y": np.array([b[3] for b in bins], dtype=np.int64),
                "ret": np.array([b[1] for b in bins]),
                "trgt": np.array([b[2] for b in bins]),
                "X": feats[t0],
                "weight": np.array(sampling.get_av_uniqueness_from_triple_barrier(spans, len(ts))),
            }
        )
    ev = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    order = np.lexsort((ev["symbol"], ev["t0_time"]))
    return {k: v[order] for k, v in ev.items()}


def fit_logit(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Weighted logistic regression with an L2 penalty on the slopes, by Newton's method."""
    Xb = np.column_stack([np.ones(len(X)), X])
    pen = np.r_[0.0, np.full(X.shape[1], L2)]
    w = w / w.mean()
    beta = np.zeros(Xb.shape[1])
    for _ in range(100):
        p = 1.0 / (1.0 + np.exp(-Xb @ beta))
        grad = Xb.T @ (w * (p - y)) + pen * beta
        hess = (Xb * (w * p * (1.0 - p))[:, None]).T @ Xb + np.diag(pen)
        step = np.linalg.solve(hess, grad)
        beta -= step
        if np.abs(step).max() < 1e-12:
            break
    return beta


def fit_fold(
    ev: dict, cols: list[int], train: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X = ev["X"][train][:, cols]
    mu, sd = X.mean(axis=0), X.std(axis=0)
    return fit_logit((X - mu) / sd, ev["y"][train].astype(float), ev["weight"][train]), mu, sd


def predict(model, X: np.ndarray) -> np.ndarray:
    beta, mu, sd = model
    return 1.0 / (1.0 + np.exp(-(beta[0] + ((X - mu) / sd) @ beta[1:])))


def oof_probability(ev: dict, walk_forward: bool = False) -> np.ndarray:
    """Out-of-fold P(profitable) for every event; NaN where walk-forward has no earlier data."""
    cols = [0, 1, 2]  # all three features: runbook 11's headline
    prob = np.full(len(ev["y"]), np.nan)
    for train, test in purged_kfold_splits(ev["t0_time"], ev["t1_time"], N_SPLITS, PCT_EMBARGO):
        if walk_forward:
            train = train[train < test.min()]
            if len(train) < 50:
                continue
        prob[test] = predict(fit_fold(ev, cols, train), ev["X"][test][:, cols])
    return prob


# ---- end of copy -----------------------------------------------------------------------------------
# The numpy CUSUM must match openquant's with a constant threshold (runbook 11's check).
_c = control[SYMBOLS[0]][1]
for _h in (0.01, 0.02, 0.04):
    assert cusum_events(
        _c, np.full(len(_c), _h)
    ).tolist() == openquant.filters.cusum_filter_indices(_c.tolist(), _h)
print("numpy CUSUM matches filters.cusum_filter_indices at constant thresholds 0.01, 0.02, 0.04")

In [ ]:
def fill(ts: list[str], points: list[tuple[str, float]]) -> np.ndarray:
    """Step function from (change point, value) rows: value held until the next change point."""
    at = {t: i for i, t in enumerate(ts)}
    pos = np.zeros(len(ts))
    for (t, value), nxt in zip(points, [*points[1:], (None, None)]):
        pos[at[t] : at[nxt[0]] if nxt[0] is not None else len(ts)] = value
    return pos


def latest(n: int, t0: np.ndarray, t1: np.ndarray, size: np.ndarray) -> np.ndarray:
    """No averaging: the position is the size of the most recently opened bet still live."""
    pos = np.zeros(n)
    for a, b, s in sorted(zip(t0.tolist(), t1.tolist(), size.tolist())):
        pos[a:b] = s  # a later bet overwrites only its own span [t0, t1)
    return pos


def event_sizes(side: np.ndarray, prob: np.ndarray | None, cfg: dict) -> np.ndarray:
    """Per-bet size before averaging and discretisation (used for `fixed`, `binary`, bet counts)."""
    if cfg["sizing"] == "fixed":
        return side.astype(float)
    if cfg["sizing"] == "binary":
        return side * (prob >= 0.5)
    if cfg["sizing"] == "oracle":  # prob holds the true regime at t0 (1 = trend): not tradable
        return side * prob
    p_bet, s_bet = sizing_inputs(side, prob, cfg)
    return np.asarray(bet_sizing.get_signal(p_bet.tolist(), cfg["k"], s_bet.tolist()))


def sizing_inputs(side: np.ndarray, prob: np.ndarray, cfg: dict) -> tuple[np.ndarray, np.ndarray]:
    """(probability of the predicted class, signed side) handed to Snippet 10.1."""
    if cfg["sizing"] == "prob-signed":
        return prob, side.astype(float)
    return np.maximum(prob, 1.0 - prob), side * (prob >= 0.5)


def positions(ts: list[str], t0, t1, side, prob, cfg: dict) -> np.ndarray:
    """Position held from the close of each bar, for one series."""
    starts, ends = [ts[i] for i in t0], [ts[i] for i in t1]
    if cfg["sizing"] in ("prob", "prob-signed"):
        p_bet, s_bet = sizing_inputs(side, prob, cfg)
        out = bet_sizing.bet_size_probability(
            starts, ends, p_bet.tolist(), s_bet.tolist(), cfg["k"], cfg["step"], cfg["avg"]
        )
        if cfg["avg"]:
            return fill(ts, out)
        return latest(len(ts), t0, t1, np.array([v for _, v in out]))
    size = event_sizes(side, prob, cfg)
    if cfg["avg"]:
        return fill(ts, bet_sizing.avg_active_signals(starts, size.tolist(), ends))
    return latest(len(ts), t0, t1, size)


def backtest(series: dict, ev: dict, prob, cfg: dict, start: dict | None = None) -> pl.DataFrame:
    """Daily gross return and turnover, averaged over symbols; `start` maps symbol -> first bar."""
    frames = []
    for sym, (ts, close) in series.items():
        mine = ev["symbol"] == sym
        p = None if prob is None else prob[mine]
        pos = positions(ts, ev["t0"][mine], ev["t1"][mine], ev["side"][mine], p, cfg)
        r = np.r_[0.0, close[1:] / close[:-1] - 1.0]
        gross = np.r_[0.0, pos[:-1] * r[1:]]  # position from the close of t-1 earns bar t
        turnover = np.abs(np.diff(np.r_[0.0, pos]))
        first = (start or {}).get(sym, int(ev["t0"][mine].min()))
        frames.append(
            pl.DataFrame(
                {
                    "ts": ts[first:],
                    "gross": gross[first:],
                    "turnover": turnover[first:],
                    "exposure": np.abs(pos)[first:],
                }
            )
        )
    return pl.concat(frames).group_by("ts").mean().sort("ts")


def net(bt: pl.DataFrame, cost_bps: float = COST_BPS) -> np.ndarray:
    return (bt["gross"] - cost_bps * 1e-4 * bt["turnover"]).to_numpy()


def sharpe(r: np.ndarray) -> float:
    """Daily Sharpe ratio; 0 for a flat strategy (no bet), which has none."""
    return 0.0 if np.ptp(r) == 0.0 else evaluation.return_moments(r.tolist()).sharpe


def max_drawdown(bt: pl.DataFrame) -> float:
    equity = np.cumprod(1.0 + net(bt))
    dd, _ = backtest_stats.drawdown_and_time_under_water(bt["ts"].to_list(), equity.tolist())
    return max(dd, default=0.0)


def evaluate(ev: dict, prob, cfg: dict, bt: pl.DataFrame) -> dict:
    r = net(bt)
    flat = np.ptp(r) == 0.0
    gross, turn = float(bt["gross"].mean()), float(bt["turnover"].mean())
    return {
        "bets": int(np.count_nonzero(event_sizes(ev["side"], prob, cfg))),
        "sharpe_net": sharpe(r),
        "sharpe_net_ann": sharpe(r) * math.sqrt(252),
        "sharpe_gross_ann": sharpe(bt["gross"].to_numpy()) * math.sqrt(252),
        "psr": float("nan") if flat else evaluation.probabilistic_sharpe_ratio(r.tolist()),
        "turnover": turn,
        "exposure": float(bt["exposure"].mean()),
        "max_dd": 0.0 if flat else max_drawdown(bt),
        "breakeven_bps": gross / turn * 1e4 if turn > 0 else float("nan"),
    }


# bet_size_probability must be get_signal -> avg_active_signals -> discrete_signal (Snippets
# 10.1-10.3, the book's order). Checked here on arbitrary bets, before any study data is used.
_rng = np.random.default_rng([SEED, 10**7])
_t0 = np.sort(_rng.choice(200, 40, replace=False))
_t1 = _t0 + _rng.integers(1, 15, 40)
_ts = business_days(260)
_p, _s = _rng.uniform(0.3, 0.9, 40), _rng.choice([-1.0, 1.0], 40)
for _k, _step in [(2, 0.0), (2, 0.1), (3, 0.2)]:
    _cfg = {"sizing": "prob", "k": _k, "step": _step, "avg": True}
    _pb, _sb = sizing_inputs(_s, _p, _cfg)
    _sig = bet_sizing.get_signal(_pb.tolist(), _k, _sb.tolist())
    _avg = bet_sizing.avg_active_signals([_ts[i] for i in _t0], _sig, [_ts[i] for i in _t1])
    _manual = bet_sizing.discrete_signal([v for _, v in _avg], _step)
    _lib = bet_sizing.bet_size_probability(
        [_ts[i] for i in _t0], [_ts[i] for i in _t1], _pb.tolist(), _sb.tolist(), _k, _step, True
    )
    assert [t for t, _ in _lib] == [t for t, _ in _avg]
    assert np.allclose([v for _, v in _lib], _manual, rtol=0.0, atol=1e-12)
print("bet_size_probability == get_signal -> avg_active_signals -> discrete_signal (3 settings)")

### The trial grid

Every configuration whose returns this notebook computes on a path is a trial on that path:

- `fixed` (primary, every bet) and `binary` (the filter), both averaged, as in runbook 11;
- `prob` for $K \in \{2, 3\}$ × step $\in \{0, 0.05, 0.1, 0.2\}$ × {`avg`, `latest`}: 16;
- `prob-signed` with $K = 2$, step 0.1, `avg`: 1;
- the filter and the headline sized rule run walk-forward: 2.

That is **21 trials**. The headline sized rule ($K = 2$, step 0.1, `avg`) was named before the run.
Runbook 11's post hoc candidate is the `prob`, $K = 2$, step 0, `avg` row. The oracle filter from
runbook 11 is shown for reference: it needs the hidden regime and is not a trial.

In [ ]:
def trial_grid() -> list[dict]:
    grid = [dict(PRIMARY), dict(FILTER)]
    for k in NUM_CLASSES:
        for step in STEP_SIZES:
            for avg in (True, False):
                grid.append(
                    {"sizing": "prob", "k": k, "step": step, "avg": avg, "cv": "purged-kfold"}
                )
    grid.append({"sizing": "prob-signed", "k": 2, "step": 0.1, "avg": True, "cv": "purged-kfold"})
    grid.append({**FILTER, "cv": "walk-forward"})
    grid.append({**HEADLINE, "cv": "walk-forward"})
    return grid


def name(cfg: dict) -> str:
    if cfg["sizing"] in ("fixed", "binary", "oracle"):
        base = {"fixed": "primary", "binary": "filter@0.5", "oracle": "oracle filter"}[
            cfg["sizing"]
        ]
    else:
        base = (
            f"{cfg['sizing']} K={cfg['k']} step={cfg['step']:g} {'avg' if cfg['avg'] else 'latest'}"
        )
    return base + (" [wf]" if cfg["cv"] == "walk-forward" else "")


GRID = trial_grid()
assert len(GRID) == 21 and len({name(g) for g in GRID}) == 21
assert HEADLINE in GRID and FILTER in GRID and PRIMARY in GRID


def run_grid(series: dict, ev: dict, probs: dict) -> tuple[list[dict], dict]:
    """Every grid configuration on one set of events; probs maps cv -> out-of-fold probabilities."""
    rows, kept = [], {}
    for trial, cfg in enumerate(GRID):
        prob, e, start = probs.get(cfg["cv"]), ev, None
        if cfg["cv"] == "walk-forward":  # score only events that have a walk-forward model
            ok = np.isfinite(prob)
            e, prob = {k: v[ok] for k, v in ev.items()}, prob[ok]
            start = {s: int(e["t0"][e["symbol"] == s].min()) for s in series}
        bt = backtest(series, e, prob, cfg, start)
        kept[name(cfg)] = (e, prob, bt)
        rows.append({"trial": trial, "config": name(cfg), **evaluate(e, prob, cfg, bt)})
    return rows, kept


def deflate(r: np.ndarray, trial_sr: list[float]) -> float:
    """DSR of one trial's net returns; NaN for a trial that is flat (no position, no Sharpe)."""
    if np.ptp(r) == 0.0:
        return float("nan")
    return evaluation.deflated_sharpe_ratio(r.tolist(), trial_sharpes=trial_sr)

## Results

### Headline path: every configuration in the registry

A fresh path (seed `[50, 0]`). All 21 configurations are recorded in a `TrialRegistry` before any
deflated Sharpe ratio is read. Sharpe ratios are annualised; `psr` is the probability that the true
Sharpe ratio is above 0; `exposure` is the mean absolute position; `turnover` the mean daily
one-way turnover; `breakeven_bps` the cost per unit turnover at which the rule stops paying.

In [ ]:
t_start = time.perf_counter()
ev_h = build_events(planted)
probs_h = {
    "purged-kfold": oof_probability(ev_h),
    "walk-forward": oof_probability(ev_h, walk_forward=True),
    None: None,
}
rows_h, kept_h = run_grid(planted, ev_h, probs_h)

PLANTED_LABEL = "planted-headline-path"
registry_planted = evaluation.TrialRegistry(REGISTRY_DIR / "planted.json")
no_bet_planted = []
for cfg, row in zip(GRID, rows_h):
    r = net(kept_h[name(cfg)][2])
    if np.ptp(r) == 0.0:  # took no bet: the registry rejects constant returns (#187)
        no_bet_planted.append(name(cfg))
        continue
    registry_planted.record({"data": PLANTED_LABEL, **cfg}, r.tolist())
sr_planted = [t.sharpe for t in registry_planted.trials] + [0.0] * len(no_bet_planted)
N_TRIALS_PLANTED = len(sr_planted)
assert N_TRIALS_PLANTED == len(GRID)

grid_h = pl.DataFrame(rows_h).with_columns(
    pl.Series("dsr", [deflate(net(kept_h[r["config"]][2]), sr_planted) for r in rows_h]).fill_nan(
        None
    )
)
if not no_bet_planted:  # the registry's own DSR must agree with the functional one
    _r = net(kept_h[name(HEADLINE)][2])
    assert math.isclose(
        registry_planted.deflated_sharpe_ratio(_r.tolist()), deflate(_r, sr_planted), rel_tol=1e-12
    )
print(
    f"events: {len(ev_h['y'])} | mean uniqueness {ev_h['weight'].mean():.3f} | "
    f"trend-regime events {regime_h[ev_h['t0']].mean():.3f} | "
    f"out-of-fold p: mean {np.mean(probs_h['purged-kfold']):.3f}, "
    f"share >= 0.5 {np.mean(probs_h['purged-kfold'] >= 0.5):.3f}"
)
print(
    f"trials: {N_TRIALS_PLANTED} ({registry_planted.n_trials} in the registry, "
    f"{len(no_bet_planted)} took no bet)"
)
print(f"registry: {registry_planted.path}", file=sys.stderr)  # machine-specific path
with pl.Config(tbl_rows=30, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(grid_h.drop("sharpe_net"))

### Headline path: sized against the filter, and the deflated Sharpe ratio

In [ ]:
def headline_row(label: str, key: str) -> dict:
    row = grid_h.filter(pl.col("config") == key).row(0, named=True)
    return {
        "strategy": label,
        **{k: row[k] for k in grid_h.columns if k not in ("trial", "config")},
    }


ev_o = ev_h
oracle_cfg = {"sizing": "oracle", "k": None, "step": None, "avg": True, "cv": None}
oracle_prob = regime_h[ev_h["t0"]].astype(float)
bt_o = backtest(planted, ev_h, oracle_prob, oracle_cfg)
headline = pl.DataFrame(
    [
        headline_row("primary (every bet, size 1)", name(PRIMARY)),
        headline_row("filter (p >= 0.5, size 1)", name(FILTER)),
        headline_row("sized (headline: K=2, step 0.1, avg)", name(HEADLINE)),
        {
            "strategy": "oracle filter (not tradable)",
            **{k: v for k, v in evaluate(ev_h, oracle_prob, oracle_cfg, bt_o).items()},
            "dsr": None,
        },
    ]
)
sr0 = evaluation.expected_max_sharpe(N_TRIALS_PLANTED, float(np.std(sr_planted)))
dsr_h = grid_h.filter(pl.col("config") == name(HEADLINE))["dsr"][0]
with pl.Config(tbl_rows=10, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(headline.drop("sharpe_net"))
print(
    f"trials: {N_TRIALS_PLANTED}; s.d. of their daily Sharpe ratios {np.std(sr_planted):.4f}; "
    f"SR0 = {sr0:.4f} daily ({sr0 * math.sqrt(252):.2f} annualised)"
)
print(f"H2 (sized DSR >= 0.95): {'supported' if dsr_h >= 0.95 else 'rejected'} (DSR {dsr_h:.3f})")
best_h = grid_h.sort("sharpe_net", descending=True).row(0, named=True)
print(
    f"best of the grid by Sharpe (post hoc): {best_h['config']}, "
    f"net Sharpe (ann.) {best_h['sharpe_net_ann']:.2f}, DSR {best_h['dsr']:.3f}"
)

In [ ]:
def draw(fig, c):
    left, right = fig.subplots(1, 2, width_ratios=[3, 2])
    for key, label, col, ls in [
        (None, "oracle filter (not tradable)", c["muted"], ":"),
        (name(HEADLINE), "sized (K=2, step 0.1, avg)", c["accent"], "-"),
        (name(FILTER), "filter (p >= 0.5)", c["text"], "-"),
        (name(PRIMARY), "primary", c["muted"], "-"),
    ]:
        bt = bt_o if key is None else kept_h[key][2]
        left.plot(np.cumsum(np.log1p(net(bt))), color=col, linestyle=ls, linewidth=1.1, label=label)
    left.axhline(0.0, color=c["rule"], linewidth=0.8)
    left.set_title("Headline path, net of 5 bps (SYNTHETIC)")
    left.set_xlabel("bar since first event")
    left.set_ylabel("cumulative log return")
    left.legend(loc="upper left")
    p = probs_h["purged-kfold"]
    grid = np.linspace(0.5, 0.95, 91)
    for k, ls in [(2, "-"), (3, "--")]:
        right.plot(
            grid,
            bet_sizing.get_signal(grid.tolist(), k),
            color=c["accent"] if k == 2 else c["muted"],
            linestyle=ls,
            linewidth=1.3,
            label=f"size, K = {k}",
        )
    right.step(
        grid,
        bet_sizing.discrete_signal(bet_sizing.get_signal(grid.tolist(), 2), 0.1),
        where="mid",
        color=c["text"],
        linewidth=0.9,
        label="K = 2, step 0.1",
    )
    twin = right.twinx()
    twin.hist(p, bins=np.linspace(0.0, 1.0, 41), color=c["rule"], alpha=0.8)
    twin.set_yticks([])
    right.set_zorder(twin.get_zorder() + 1)
    right.patch.set_visible(False)
    right.set_xlim(0.0, 1.0)
    right.set_title("Snippet 10.1 and the out-of-fold p")
    right.set_xlabel("meta-model probability p")
    right.set_ylabel("bet size")
    right.legend(loc="upper left")


nbfigures.figure(
    "nb13-headline",
    draw,
    size=(7.6, 3.3),
    alt="Left: cumulative net log return on the fresh planted-signal path for the primary crossover, the flat 0.5 meta filter, the probability-sized rule and the oracle filter. Right: the Snippet 10.1 bet-size curves for two and three classes and the stepped two-class curve, over a histogram of the meta-model's out-of-fold probabilities.",
)

### Monte Carlo: 30 fresh paths per signal strength

Every grid configuration on fresh paths (seeds `[50, strength index, path]`), for
$\kappa_{\text{trend}} \in \{0, 0.12, 0.25\}$ with $\kappa_{\text{chop}} = -0.4\,\kappa_{\text{trend}}$.
$\kappa = 0$ is the zero-signal control. On each path, each configuration's DSR is deflated by all
21 trials on that path. These paths test the hypotheses; they select nothing, so they add no trials
to the headline path.

In [ ]:
COST_KEYS = (name(PRIMARY), name(FILTER), name(HEADLINE))


def one_path(k: int, rep: int) -> list[dict]:
    kappa = STRENGTHS[k]
    close, regime = simulate([SEED, k + 1, rep], kappa, -0.4 * kappa)
    series = {"PLANTED": (TS, close)}
    ev = build_events(series)
    probs = {
        "purged-kfold": oof_probability(ev),
        "walk-forward": oof_probability(ev, walk_forward=True),
        None: None,
    }
    rows, kept = run_grid(series, ev, probs)
    trial_sr = [r["sharpe_net"] for r in rows]  # flat trials already count as 0 (see sharpe())
    for r in rows:
        bt = kept[r["config"]][2]
        r["dsr"] = deflate(net(bt), trial_sr)
        r["gross_mean"] = float(bt["gross"].mean())
        if r["config"] in COST_KEYS:  # Sharpe ratio re-priced at each cost (numpy, ddof=1)
            for c in COST_GRID_BPS:
                x = net(bt, c)
                r[f"sr_{c:g}bps"] = float(x.mean() / x.std(ddof=1) * math.sqrt(252))
    oracle_prob = regime[ev["t0"]].astype(float)
    bt = backtest(series, ev, oracle_prob, oracle_cfg)
    rows.append(
        {
            "trial": -1,
            "config": "oracle filter",
            **evaluate(ev, oracle_prob, oracle_cfg, bt),
            "dsr": float("nan"),
            "gross_mean": float(bt["gross"].mean()),
        }
    )
    return [{"kappa": kappa, "rep": rep, "events": len(ev["y"]), **r} for r in rows]


t_mc = time.perf_counter()
mc = pl.DataFrame(
    [row for k in range(len(STRENGTHS)) for rep in range(N_REPS) for row in one_path(k, rep)],
    infer_schema_length=None,
)
print(f"Monte Carlo: {mc.height} rows", file=sys.stderr)
print(f"Monte Carlo time: {time.perf_counter() - t_mc:.0f} s", file=sys.stderr)


def wide(metric: str) -> pl.DataFrame:
    return mc.pivot(on="config", index=["kappa", "rep"], values=metric).sort("kappa", "rep")


def paired(a: str, b: str, metric: str = "sharpe_net_ann") -> pl.DataFrame:
    w, rows = wide(metric), []
    for kappa in STRENGTHS:
        g = w.filter(pl.col("kappa") == kappa)
        d = (g[a] - g[b]).to_numpy()
        rows.append(
            {
                "kappa": kappa,
                "mean_diff": float(d.mean()),
                "t": float(d.mean() / (d.std(ddof=1) / math.sqrt(len(d)))),
                "share_of_paths_up": float((d > 0).mean()),
            }
        )
    return pl.DataFrame(rows)


tests = pl.concat(
    [
        paired(name(HEADLINE), name(FILTER)).with_columns(test=pl.lit("H1/H3a sized - filter")),
        paired(name(HEADLINE), name(PRIMARY)).with_columns(test=pl.lit("sized - primary")),
        paired(name(FILTER), name(PRIMARY)).with_columns(test=pl.lit("filter - primary")),
        paired(name({**HEADLINE, "step": 0.0}), name(FILTER)).with_columns(
            test=pl.lit("rb11 candidate (step 0) - filter")
        ),
        paired(
            name({**HEADLINE, "cv": "walk-forward"}), name({**FILTER, "cv": "walk-forward"})
        ).with_columns(test=pl.lit("walk-forward: sized - filter")),
        paired("oracle filter", name(PRIMARY)).with_columns(test=pl.lit("oracle - primary")),
    ]
).select("test", "kappa", "mean_diff", "t", "share_of_paths_up")

show = [name(PRIMARY), name(FILTER), name(HEADLINE), "oracle filter"]
summary = (
    mc.filter(pl.col("config").is_in(show))
    .group_by("kappa", "config")
    .agg(
        pl.col("bets").mean(),
        pl.col("sharpe_net_ann").mean(),
        pl.col("turnover").mean(),
        pl.col("exposure").mean(),
        pl.col("max_dd").mean(),
        (pl.col("gross_mean").sum() / pl.col("turnover").sum() * 1e4).alias("breakeven_bps"),
        (pl.col("dsr") >= 0.95).mean().alias("share_dsr_ge_0.95"),
    )
    .sort("kappa", pl.col("config").replace_strict({c: i for i, c in enumerate(show)}))
)
with pl.Config(tbl_rows=40, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(summary)
    print(tests)

In [ ]:
# Per path: does the best of the 21 configurations (picked after the fact) clear DSR 0.95?
best_per_path = (
    mc.filter(pl.col("trial") >= 0)
    .sort("sharpe_net", descending=True)
    .group_by("kappa", "rep", maintain_order=True)
    .first()
    .sort("kappa", "rep")
)
fd = (
    best_per_path.group_by("kappa")
    .agg(
        (pl.col("dsr") >= 0.95).sum().alias("paths_best_dsr_ge_0.95"),
        pl.len().alias("paths"),
        pl.col("dsr").median().alias("median_best_dsr"),
        pl.col("config").mode().first().alias("most_often_best"),
    )
    .sort("kappa")
)
headline_fd = (
    mc.filter(pl.col("config") == name(HEADLINE))
    .group_by("kappa")
    .agg((pl.col("dsr") >= 0.95).sum().alias("paths_sized_dsr_ge_0.95"))
    .sort("kappa")
)
fd = fd.join(headline_fd, on="kappa")


def binom_sf(k: int, n: int, p: float) -> float:
    """P(X >= k) for X ~ Binomial(n, p)."""
    return sum(math.comb(n, i) * p**i * (1 - p) ** (n - i) for i in range(k, n + 1))


fd = fd.with_columns(
    pl.struct("paths_best_dsr_ge_0.95", "paths")
    .map_elements(
        lambda s: binom_sf(s["paths_best_dsr_ge_0.95"], s["paths"], 0.05), return_dtype=pl.Float64
    )
    .alias("p_value_vs_5pct")
)
with pl.Config(tbl_rows=10, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(fd)

In [ ]:
def draw(fig, c):
    left, right = fig.subplots(1, 2)
    w = wide("sharpe_net_ann")
    for ax, (a, b, title) in zip(
        (left, right),
        [
            (name(HEADLINE), name(FILTER), "Net Sharpe (ann.): sized - filter"),
            (name(FILTER), name(PRIMARY), "Net Sharpe (ann.): filter - primary"),
        ],
    ):
        data = [
            (w.filter(pl.col("kappa") == k)[a] - w.filter(pl.col("kappa") == k)[b]).to_numpy()
            for k in STRENGTHS
        ]
        ax.boxplot(
            data,
            positions=range(len(STRENGTHS)),
            widths=0.5,
            patch_artist=True,
            boxprops={"facecolor": c["surface"], "edgecolor": c["muted"]},
            medianprops={"color": c["accent"], "linewidth": 1.6},
            whiskerprops={"color": c["muted"]},
            capprops={"color": c["muted"]},
            flierprops={"markeredgecolor": c["muted"], "markersize": 3},
        )
        ax.axhline(0.0, color=c["rule"], linewidth=0.9)
        ax.set_xticks(range(len(STRENGTHS)), [f"κ = {k}" for k in STRENGTHS])
        ax.set_title(title)
    left.set_ylabel(f"difference across {N_REPS} paths")


nbfigures.figure(
    "nb13-monte-carlo",
    draw,
    size=(7.6, 3.3),
    alt="Box plots over simulated paths at signal strengths 0, 0.12 and 0.25. Left: the net Sharpe ratio of the probability-sized rule minus that of the flat 0.5 filter. Right: the flat filter minus the unfiltered primary model.",
)

### Every configuration across paths, and cost sensitivity

Mean over the 30 paths at each strength of each grid configuration's net Sharpe ratio, and of the
headline rules' net Sharpe ratio as the cost per unit turnover rises from 0 to 40 bps. The cost
sweep re-prices the same positions; it selects nothing.

In [ ]:
grid_mc = (
    mc.filter(pl.col("trial") >= 0)
    .group_by("config", "kappa")
    .agg(pl.col("sharpe_net_ann").mean(), pl.col("turnover").mean())
    .pivot(on="kappa", index="config", values=["sharpe_net_ann", "turnover"])
)
order = {name(g): i for i, g in enumerate(GRID)}
grid_mc = grid_mc.sort(pl.col("config").replace_strict(order))
with pl.Config(tbl_rows=30, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(grid_mc)


def cost_curve(kappa: float, key: str) -> list[float]:
    """Mean over paths of the annualised net Sharpe ratio at each cost in COST_GRID_BPS."""
    g = mc.filter((pl.col("kappa") == kappa) & (pl.col("config") == key))
    return [float(g[f"sr_{c:g}bps"].mean()) for c in COST_GRID_BPS]


# The numpy re-pricing at 5 bps must agree with openquant.evaluation's Sharpe ratio.
_g = mc.filter(pl.col("config").is_in(COST_KEYS))
assert np.allclose(_g[f"sr_{COST_BPS:g}bps"], _g["sharpe_net_ann"], rtol=1e-9, atol=1e-12)


costs = pl.DataFrame(
    [
        {
            "kappa": kappa,
            "config": key,
            **{f"{c:g}bps": v for c, v in zip(COST_GRID_BPS, cost_curve(kappa, key))},
        }
        for kappa in STRENGTHS
        for key in (name(PRIMARY), name(FILTER), name(HEADLINE))
    ]
)
with pl.Config(tbl_rows=20, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(costs)

In [ ]:
def draw(fig, c):
    ax = fig.subplots()
    for key, label, col, ls in [
        (name(PRIMARY), "primary", c["muted"], "-"),
        (name(FILTER), "filter (p >= 0.5)", c["text"], "-"),
        (name(HEADLINE), "sized (K=2, step 0.1, avg)", c["accent"], "-"),
    ]:
        ax.plot(
            COST_GRID_BPS,
            cost_curve(KAPPA_TREND, key),
            color=col,
            linestyle=ls,
            marker="o",
            markersize=3.5,
            linewidth=1.2,
            label=label,
        )
    ax.axhline(0.0, color=c["rule"], linewidth=0.9)
    ax.axvline(COST_BPS, color=c["rule"], linewidth=0.9, linestyle=":")
    ax.set_title(f"Net Sharpe vs cost, κ = {KAPPA_TREND}, mean of {N_REPS} paths (SYNTHETIC)")
    ax.set_xlabel("cost per unit of one-way turnover (bps)")
    ax.set_ylabel("net Sharpe ratio (ann.)")
    ax.legend(loc="upper right")


nbfigures.figure(
    "nb13-cost-sensitivity",
    draw,
    alt="Mean net Sharpe ratio over simulated paths at the headline signal strength for the primary model, the flat 0.5 filter and the probability-sized rule, as the cost per unit of turnover rises from 0 to 40 basis points.",
)

### Control: the SYNTHETIC `fetch` sample

The same 21 configurations on the five `SYN_*` symbols pooled, recorded in a registry of their own
(`OPENQUANT_RUNBOOK_REGISTRY` if set, so that a run on your data accumulates its trials). These are
random walks: the right answer is no edge anywhere.

In [ ]:
registry_control = evaluation.TrialRegistry(
    Path(_reg_env) if _reg_env else REGISTRY_DIR / "control.json"
)
CONTROL_LABEL = f"control-{meta['dataset_hash'][7:19]}"
ev_c = build_events(control)
probs_c = {
    "purged-kfold": oof_probability(ev_c),
    "walk-forward": oof_probability(ev_c, walk_forward=True),
    None: None,
}
rows_c, kept_c = run_grid(control, ev_c, probs_c)
no_bet_control = []
for cfg in GRID:
    r = net(kept_c[name(cfg)][2])
    if np.ptp(r) == 0.0:  # took no bet: the registry rejects constant returns (#187)
        no_bet_control.append(name(cfg))
        continue
    registry_control.record({"data": CONTROL_LABEL, **cfg}, r.tolist())
# A persisted registry may hold earlier sessions' trials too: all of them count.
sr_control = [t.sharpe for t in registry_control.trials] + [0.0] * len(no_bet_control)
grid_c = pl.DataFrame(rows_c).with_columns(
    pl.Series(
        "dsr",
        [deflate(net(kept_c[r["config"]][2]), sr_control) for r in rows_c],
    ).fill_nan(None)
)
dsr_c = {
    "sized": grid_c.filter(pl.col("config") == name(HEADLINE))["dsr"][0],
    "best": grid_c.sort("sharpe_net", descending=True)["dsr"][0],
}
print(
    f"{LABEL} control: {len(ev_c['y'])} events over {len(SYMBOLS)} symbols; "
    f"trials {len(sr_control)} ({registry_control.n_trials} in the registry, "
    f"{len(no_bet_control)} took no bet)"
)
print(f"registry: {registry_control.path}", file=sys.stderr)
with pl.Config(tbl_rows=30, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(grid_c.drop("sharpe_net"))
print(
    f"DSR: sized {dsr_c['sized'] or float('nan'):.3f}, best of grid {dsr_c['best'] or float('nan'):.3f}"
)

### Illustration (not a hypothesis): dynamic sizing and limit prices (Snippet 10.4)

§10.6 sizes from a *price forecast* instead of a probability. The meta-model gives a probability,
so one way to get a forecast is the expected price at the first barrier:
$f = P_{t_0} \exp\!\big(s\,(2p - 1)\cdot 1.5\,\sigma_{t_0}\big)$, ignoring timeouts. The sigmoid is
calibrated per event so that a divergence of one full barrier ($1.5\,\sigma_{t_0} P_{t_0}$) is a
0.95 bet (`get_w`), then `get_target_pos` gives the position out of 100 units, and `limit_price` the
worst average price at which building it from flat is still justified (the #163-corrected
formula). `bet_size_dynamic` is not used because it fixes the calibration at 10 price units (see
the `bet_sizing` page). No returns are computed from this.

In [ ]:
sigma_h = daily_vol(TS, close_h)
p_h = probs_h["purged-kfold"]
rows = []
for i in range(len(ev_h["y"]) - 6, len(ev_h["y"])):
    t0, s, p = int(ev_h["t0"][i]), float(ev_h["side"][i]), float(p_h[i])
    m_p, barrier = float(close_h[t0]), PT_SL * float(sigma_h[t0])
    f = m_p * math.exp(s * (2 * p - 1) * barrier)
    w = bet_sizing.get_w(barrier * m_p, 0.95, "sigmoid")
    target = bet_sizing.get_target_pos(w, f, m_p, 100.0, "sigmoid")
    limit = bet_sizing.limit_price(target, 0.0, f, w, 100.0, "sigmoid") if target else float("nan")
    rows.append(
        {
            "t0": TS[t0][:10],
            "side": s,
            "p": p,
            "market": m_p,
            "forecast": f,
            "target_of_100": target,
            "limit": limit,
            "snippet_10.1_x100": 100 * event_sizes(np.array([s]), np.array([p]), HEADLINE)[0],
        }
    )
with pl.Config(tbl_rows=10, float_precision=3, tbl_width_chars=220, tbl_cols=20):
    print(pl.DataFrame(rows))

### Look-ahead and leakage checks

1. **Features, sides, volatility and events use only the past** (runbook 11's check 1).
2. **A label uses only its own span** (runbook 11's check 2, on 40 events).
3. **Purging and embargo** (runbook 11's check 3).
4. **The meta-model never sees the test fold** (runbook 11's check 4).
5. **Sizing uses only bets already open.** For 20 cut bars $t^*$: re-drawing the probability of every
   bet opened after $t^*$, and moving the end $t_1$ of every bet still live at $t^*$ to a random
   later bar, leaves every position up to and including $t^*$ bit-identical, for every `prob`
   configuration. Re-drawing the probability of one bet live at $t^*$ must change the position (so
   the test can fail).
6. **Shift test.** The position set at the close of $t$ earns bar $t + 1$. Deliberately trading
   one bar early ($w_t r_t$, which peeks) must inflate the Sharpe ratio, and one bar late must not
   destroy it: a slow signal loses little to a one-bar delay, while a leak would collapse.
7. **Shuffle test.** Permuting the probabilities across events ($N = 200$) keeps their
   distribution but breaks their link to the bets. The sized rule's gain over the filter must come
   from that link, so the real headline Sharpe ratio should sit in the upper tail of the permuted
   ones.

In [ ]:
rng = np.random.default_rng([SEED, 10**6])
cut = N_BARS // 2
shocked = close_h.copy()
shocked[cut + 1 :] = close_h[cut] * np.exp(np.cumsum(rng.normal(0.0, 0.03, N_BARS - cut - 1)))
side_a, feat_a = primary_and_features(close_h)
side_b, feat_b = primary_and_features(shocked)
vol_a, vol_b = daily_vol(TS, close_h), daily_vol(TS, shocked)
thr_a, thr_b = CUSUM_MULT * np.r_[np.nan, vol_a[:-1]], CUSUM_MULT * np.r_[np.nan, vol_b[:-1]]
ev_a, ev_b = cusum_events(close_h, thr_a), cusum_events(shocked, thr_b)
check1 = {
    "side": np.array_equal(side_a[: cut + 1], side_b[: cut + 1], equal_nan=True),
    "features": np.array_equal(feat_a[: cut + 1], feat_b[: cut + 1], equal_nan=True),
    "daily vol": np.array_equal(vol_a[: cut + 1], vol_b[: cut + 1], equal_nan=True),
    "events": ev_a[ev_a <= cut].tolist() == ev_b[ev_b <= cut].tolist(),
    "the future did change": not np.array_equal(
        feat_a[cut + 1 :], feat_b[cut + 1 :], equal_nan=True
    ),
}
print("1. unchanged up to the cut:", check1)
assert all(check1.values())

unchanged, changed_inside = 0, 0
for i in rng.choice(len(ev_h["y"]), 40, replace=False):
    a, b = int(ev_h["t0"][i]), int(ev_h["t1"][i])
    for lo, hi, what in [(b + 1, N_BARS, "after"), (a + 1, b + 1, "inside")]:
        path = close_h.copy()
        path[lo:hi] = close_h[lo - 1] * np.exp(np.cumsum(rng.normal(0.0, 0.03, hi - lo)))
        vb = labeling.add_vertical_barrier([TS[a]], TS, path.tolist(), num_days=HORIZON_DAYS)
        e = labeling.triple_barrier_events(
            TS,
            path.tolist(),
            [TS[a]],
            [TS[a]],
            [float(sigma_h[a])],
            pt=PT_SL,
            sl=PT_SL,
            vertical_barrier_times=vb,
            side_prediction=[(TS[a], float(ev_h["side"][i]))],
        )
        label = labeling.get_bins(e, TS, path.tolist())[0][3]
        same = e[0][1] == TS[b] and label == ev_h["y"][i]
        if what == "after":
            unchanged += same
        else:
            changed_inside += not same
print(
    f"2. labels unchanged when the future after t1 changes: {unchanged} of 40; "
    f"changed when (t0, t1] changes: {changed_inside} of 40"
)
assert unchanged == 40 and changed_inside > 0

folds = split_with_diagnostics(ev_h["t0_time"], ev_h["t1_time"], N_SPLITS, PCT_EMBARGO)
overlaps = [
    count_train_test_overlaps(
        ev_h["t0_time"], ev_h["t1_time"], f["train_indices"], f["test_indices"]
    )
    for f in folds
]
embargo_trained = [len(np.intersect1d(f["embargo_indices"], f["train_indices"])) for f in folds]
naive = [
    count_train_test_overlaps(ev_h["t0_time"], ev_h["t1_time"], tr, te)
    for tr, te in naive_kfold_splits(len(ev_h["y"]), N_SPLITS)
]
print(
    f"3. purged: train/test overlaps per fold {overlaps}, embargoed events trained on "
    f"{embargo_trained}; unpurged k-fold overlaps {naive}"
)
assert sum(overlaps) == 0 and sum(embargo_trained) == 0 and sum(naive) > 0

identical = 0
for train, test in purged_kfold_splits(ev_h["t0_time"], ev_h["t1_time"], N_SPLITS, PCT_EMBARGO):
    noisy = {**ev_h, "X": ev_h["X"].copy(), "y": ev_h["y"].copy()}
    noisy["X"][test] = rng.normal(size=noisy["X"][test].shape)
    noisy["y"][test] = rng.integers(0, 2, len(test))
    a_model, b_model = fit_fold(ev_h, [0, 1, 2], train), fit_fold(noisy, [0, 1, 2], train)
    identical += all(np.array_equal(x, y) for x, y in zip(a_model, b_model))
print(
    f"4. folds whose fitted model is bit-identical after the test fold is replaced with noise: "
    f"{identical} of {N_SPLITS}"
)
assert identical == N_SPLITS

In [ ]:
t0_h, t1_h, side_h = ev_h["t0"], ev_h["t1"], ev_h["side"]
prob_cfgs = [g for g in GRID if g["sizing"].startswith("prob") and g["cv"] == "purged-kfold"]
cuts = np.sort(rng.choice(np.arange(int(t0_h.min()) + 1, N_BARS - 30), 20, replace=False))
same, moved = 0, 0
for cut in cuts.tolist():
    later = t0_h > cut
    live = (t0_h <= cut) & (t1_h > cut)
    p_alt = p_h.copy()
    p_alt[later] = rng.uniform(0.0, 1.0, int(later.sum()))
    t1_alt = t1_h.copy()
    t1_alt[live] = np.minimum(N_BARS - 1, cut + 1 + rng.integers(0, 60, int(live.sum())))
    for cfg in prob_cfgs:
        a = positions(TS, t0_h, t1_h, side_h, p_h, cfg)
        b = positions(TS, t0_h, t1_alt, side_h, p_alt, cfg)
        same += np.array_equal(a[: cut + 1], b[: cut + 1])
    # Counter-check: re-drawing the probability of a live bet must move the position at the cut.
    if live.any():
        j = np.flatnonzero(live)[-1]
        p_one = p_h.copy()
        p_one[j] = 0.99 if p_h[j] < 0.75 else 0.51
        cfg = {**HEADLINE, "step": 0.0}
        moved += (
            positions(TS, t0_h, t1_h, side_h, p_one, cfg)[cut]
            != positions(TS, t0_h, t1_h, side_h, p_h, cfg)[cut]
        )
n_live = sum(bool(((t0_h <= c) & (t1_h > c)).any()) for c in cuts.tolist())
print(
    f"5. positions up to the cut bit-identical: {same} of {len(cuts) * len(prob_cfgs)} "
    f"(20 cuts x {len(prob_cfgs)} configurations); one live bet re-drawn moved the position at "
    f"{moved} of {n_live} cuts that had a live bet"
)
assert same == len(cuts) * len(prob_cfgs) and moved == n_live and n_live > 0

bt_hl = kept_h[name(HEADLINE)][2]
pos_hl = positions(TS, t0_h, t1_h, side_h, p_h, HEADLINE)
r_all = np.r_[0.0, close_h[1:] / close_h[:-1] - 1.0]
first = int(t0_h.min())
turn = np.abs(np.diff(np.r_[0.0, pos_hl]))
cost = COST_BPS * 1e-4 * turn


def ann(gross: np.ndarray) -> float:
    return sharpe((gross - cost)[first:]) * math.sqrt(252)


shift = {
    "as run (w[t-1] r[t])": ann(np.r_[0.0, pos_hl[:-1] * r_all[1:]]),
    "one bar early, peeks (w[t] r[t])": ann(pos_hl * r_all),
    "one bar late (w[t-2] r[t])": ann(np.r_[0.0, 0.0, pos_hl[:-2] * r_all[2:]]),
}
assert math.isclose(shift["as run (w[t-1] r[t])"], sharpe(net(bt_hl)) * math.sqrt(252))
print("6. shift test, sized net Sharpe (ann.):", {k: round(v, 2) for k, v in shift.items()})
assert shift["one bar early, peeks (w[t] r[t])"] > shift["as run (w[t-1] r[t])"] + 0.5

filter_sr = sharpe(net(kept_h[name(FILTER)][2])) * math.sqrt(252)
perm = []
for _ in range(N_SHUFFLES):
    p_perm = rng.permutation(p_h)
    perm.append(sharpe(net(backtest(planted, ev_h, p_perm, HEADLINE))) * math.sqrt(252))
perm = np.array(perm)
real = shift["as run (w[t-1] r[t])"]
print(
    f"7. shuffle test: sized net Sharpe (ann.) {real:.2f} vs permuted probabilities: mean "
    f"{perm.mean():.2f}, 95th pct {np.quantile(perm, 0.95):.2f}; share of permutations >= real "
    f"{(perm >= real).mean():.3f}; filter {filter_sr:.2f}"
)

## Analysis

In [ ]:
h = {r["strategy"]: r for r in headline.iter_rows(named=True)}
print(
    "headline path, net Sharpe (ann.): "
    + ", ".join(f"{k.split(' (')[0]} {v['sharpe_net_ann']:.2f}" for k, v in h.items())
)


def verdict(test: str, kappa: float) -> dict:
    return tests.filter((pl.col("test") == test) & (pl.col("kappa") == kappa)).row(0, named=True)


h1 = verdict("H1/H3a sized - filter", KAPPA_TREND)
h3a = verdict("H1/H3a sized - filter", 0.0)
fd0 = fd.filter(pl.col("kappa") == 0.0).row(0, named=True)
print(
    f"H1 (sized > filter, kappa={KAPPA_TREND}): mean diff {h1['mean_diff']:+.3f}, "
    f"t = {h1['t']:.2f} -> {'supported' if h1['t'] > 1.645 else 'rejected'}"
)
print(
    f"H2 (headline-path sized DSR >= 0.95): {'supported' if dsr_h >= 0.95 else 'rejected'} (DSR {dsr_h:.3f})"
)
print(
    f"H3a (kappa=0, sized not > filter): mean diff {h3a['mean_diff']:+.3f}, t = {h3a['t']:.2f} -> "
    f"{'passes' if h3a['t'] < 1.645 else 'FALSE POSITIVE'}"
)
print(
    f"H3b (kappa=0, best-of-grid DSR >= 0.95 on <= 5% of paths): {fd0['paths_best_dsr_ge_0.95']} of "
    f"{fd0['paths']} (p = {fd0['p_value_vs_5pct']:.3f}) -> "
    f"{'passes' if fd0['p_value_vs_5pct'] >= 0.05 else 'FALSE DISCOVERY'}"
)
h3c = all(v is None or v < 0.95 for v in dsr_c.values())
print(
    f"H3c ({LABEL} control, sized and best-of-grid DSR < 0.95): sized {dsr_c['sized'] or float('nan'):.3f}, "
    f"best {dsr_c['best'] or float('nan'):.3f} -> {'passes' if h3c else 'FALSE DISCOVERY'}"
)

ANALYSIS_PLACEHOLDER

## Promotion decision

DECISION_PLACEHOLDER

CHECKLIST_PLACEHOLDER

## Reproducibility

In [ ]:
def git_sha() -> str:
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
        ).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"


manifest = openquant.research.research_run_manifest(
    {
        "notebook": "13_bet_sizing_from_probabilities",
        "seed": SEED,
        "generator": {
            "n_bars": N_BARS,
            "p_stay": P_STAY,
            "vol": [VOL_TREND, VOL_CHOP],
            "logvol": [LOGVOL_AR, LOGVOL_SD],
            "mom_len": MOM_LEN,
            "kappa": [KAPPA_TREND, KAPPA_CHOP],
            "strengths": list(STRENGTHS),
            "reps": N_REPS,
        },
        "pipeline": {
            "vol_span": VOL_SPAN,
            "cusum_mult": CUSUM_MULT,
            "pt_sl": PT_SL,
            "horizon_days": HORIZON_DAYS,
            "ma": [FAST, SLOW],
            "n_splits": N_SPLITS,
            "pct_embargo": PCT_EMBARGO,
            "l2": L2,
            "features": list(FEATURES),
        },
        "sizing": {
            "cost_bps": COST_BPS,
            "cost_grid_bps": list(COST_GRID_BPS),
            "num_classes": list(NUM_CLASSES),
            "step_sizes": list(STEP_SIZES),
            "headline": HEADLINE,
            "filter": FILTER,
            "shuffles": N_SHUFFLES,
        },
        "symbols": SYMBOLS,
        "trials": {
            "planted": N_TRIALS_PLANTED,
            "control": len(sr_control),
            "monte_carlo_paths": N_REPS * len(STRENGTHS),
            "per_monte_carlo_path": len(GRID),
        },
    }
)
openquant.data.record_dataset_hash(
    manifest,
    digest=meta["dataset_hash"],
    **{k: meta[k] for k in ("source", "symbols", "start", "end", "rows")},
)
manifest["planted_hash"] = planted_hash

print("package:      pyopenquant", version("pyopenquant"))
print("data hash:   ", manifest["dataset_hash"], f"({LABEL} control)")
print("seed:        ", SEED)
print("trials:      ", manifest["config"]["trials"])
print("config:      ", manifest["config_digest"])
# Platform- or commit-dependent values go to stderr, which the staleness check ignores.
print("planted:     ", planted_hash, file=sys.stderr)  # float bits can differ by platform
print("numpy:       ", np.__version__, "| polars:", pl.__version__, file=sys.stderr)
print("git sha:     ", git_sha(), file=sys.stderr)
print(f"run time:     {time.perf_counter() - t_start:.0f} s", file=sys.stderr)